In [40]:
import numpy as np
import numpy.typing as npt
from astropy.time import Time
from sorts.propagator import SGP4
from sorts.space_object import SpaceObject
from sorts.radar.radars import get_radar
from sorts.types import Datetime64_us, Timedelta64_us, Float64_as_sec
from sorts.controller_v2.tracker_controller import TrackerController
from sorts.controller_v2.fence_scan_controller_new import FenceScanController
from sorts.schedule_v2 import Schedule, ExperimentDetail
from sorts.scheduler_v2.priority_scheduling import _priority_scheduling_df

# import for plottings
from IPython.display import display
import pandas as pd
import ipywidgets as widgets
from sorts import plots

In [ ]:
# disable pandas table wrapping
pd.set_option("display.expand_frame_repr", False)

# activate Bokeh output in Jupyter notebook
from bokeh.io import output_notebook, push_notebook
output_notebook()

Loading BokehJS ...

In [3]:
epoch = Time(53005.0, format="mjd", scale="utc")  # 2004-01-01 00:00:00Z

# the first set of value used, not much use now; kept for ref
# start_time = Time("2025-06-30 00:00:00")
# end_time = Time("2025-06-30 00:00:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

# a 1 sec long period, the sbobj should be very close to right up ahead of eiscat3d tx-0 station
# start_time = Time("2025-01-01 04:04:00")
# end_time = Time("2025-01-01 04:04:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

# an extended duration which expands around from the 1 sec period above
# the `control_slice_duration` is much longer than normal, practical radar `control_slice_duration`
# for easier debugging, inspection of scheduling/schedules
start_time = Time("2025-01-01 02:45:00")
end_time = Time("2025-01-01 06:15:00")
control_slice_duration = np.timedelta64(int(60 * 1e6), "us")

# same as above, but use more realistic 10ms `control_slice_duration`
# start_time = Time("2025-01-01 02:45:00")
# end_time = Time("2025-01-01 06:15:00")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

eiscat3d = get_radar("eiscat3d", "stage1-array")

spobj = SpaceObject(
    SGP4,
    propagator_options={"settings": {"out_frame": "ITRF"}},
    a=7200e3,
    e=0.02,
    i=75,
    raan=86,
    aop=0,
    mu0=60,
    epoch=epoch,
    parameters={"d": 0.1},
)


exp_detail_0 = ExperimentDetail(
    id=0,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

exp_detail_1 = ExperimentDetail(
    id=1,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

time_arr: npt.NDArray[Datetime64_us] = np.arange(
    start_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    end_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    control_slice_duration,
)
time_arr = time_arr[::4] # TODO: remove; strided to bring up the effects of scheduling
dt_arr: npt.NDArray[Timedelta64_us] = time_arr - epoch.to_value("datetime64").astype("datetime64[us]")  # type: ignore
dsec_arr: npt.NDArray[Float64_as_sec] = dt_arr.astype(np.float64) / 1e6  # type: ignore

ecefs = spobj.get_state(dsec_arr)

trackerController = TrackerController(
    tx_station=eiscat3d.tx[0],
    rx_stations=[],
    time=time_arr,
    space_object_states=ecefs,
    exp_detail=exp_detail_0,
    min_elevation=10,
)

fenceScanController = FenceScanController(
    tx_station=eiscat3d.tx[0],
    rx_station=[],
    exp_datail=exp_detail_1,
    azimuth=90, # sweep from east to west
    min_elevation=30,
    pointings_per_cycle=40,
)

In [4]:
ecefs.shape

(6, 53)

In [5]:
# plots.ecef_states_positions_plot(ecefs)

In [6]:
tracker_schs = trackerController.generate()
tracker_tx_sch_df = tracker_schs.tx_schedule.as_dataframe()
tracker_tx_sch_df

,stt_tstmp_us,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,17.942607,14.412365,0,2025-01-01 02:46:00
1,2025-01-01 03:45:00,-152.513503,17.085825,0,2025-01-01 03:46:00
2,2025-01-01 03:49:00,-153.030311,31.502623,0,2025-01-01 03:50:00
3,2025-01-01 03:53:00,-153.881047,45.803195,0,2025-01-01 03:54:00
4,2025-01-01 03:57:00,-155.744935,59.942156,0,2025-01-01 03:58:00
5,2025-01-01 04:01:00,-161.477631,73.810823,0,2025-01-01 04:02:00
6,2025-01-01 04:05:00,148.242673,85.818447,0,2025-01-01 04:06:00
7,2025-01-01 04:09:00,50.677598,77.437526,0,2025-01-01 04:10:00
8,2025-01-01 04:13:00,42.266485,64.070795,0,2025-01-01 04:14:00
9,2025-01-01 04:17:00,40.127128,50.600575,0,2025-01-01 04:18:00


In [7]:
# check schedule df memory usage (MB)
tracker_tx_sch_df.memory_usage().sum()/1e6

np.float64(0.001088)

In [8]:
fence_schs = fenceScanController.generate(start_time, end_time)
fence_tx_sch_df = fence_schs.tx_schedule.as_dataframe()
fence_tx_sch_df

,stt_tstmp_us,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,90.0,30.000000,1,2025-01-01 02:46:00
1,2025-01-01 02:46:00,90.0,33.076923,1,2025-01-01 02:47:00
2,2025-01-01 02:47:00,90.0,36.153846,1,2025-01-01 02:48:00
3,2025-01-01 02:48:00,90.0,39.230769,1,2025-01-01 02:49:00
4,2025-01-01 02:49:00,90.0,42.307692,1,2025-01-01 02:50:00
...,...,...,...,...,...
205,2025-01-01 06:10:00,90.0,45.384615,1,2025-01-01 06:11:00
206,2025-01-01 06:11:00,90.0,48.461538,1,2025-01-01 06:12:00
207,2025-01-01 06:12:00,90.0,51.538462,1,2025-01-01 06:13:00
208,2025-01-01 06:13:00,90.0,54.615385,1,2025-01-01 06:14:00


In [9]:
# check schedule df memory usage (MB)
fence_tx_sch_df.memory_usage().sum()/1e6

np.float64(0.008528)

In [10]:
tx_sch = tracker_schs.tx_schedule
# plots.azel_polar_plot(tx_sch.pointing_az, tx_sch.pointing_el)

In [11]:
tx_sch = fence_schs.tx_schedule
# plots.azel_polar_plot(tx_sch.pointing_az, tx_sch.pointing_el)

In [12]:
(tracker_schs.tx_schedule.meta, fence_schs.tx_schedule.meta)

({0: ExperimentDetail(id=0, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(60000000,'us'))},
 {1: ExperimentDetail(id=1, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(60000000,'us'))})

In [13]:
master_sch_meta = {0: exp_detail_0, 1: exp_detail_1}
master_sch_df = _priority_scheduling_df([tracker_schs.tx_schedule, fence_schs.tx_schedule])
master_sch_df

,stt_tstmp_us,pointing_az,pointing_el,exp_num,end_time,allowed_start_time,allowed_end_time,is_overlaped
0,2025-01-01 02:45:00,17.942607,14.412365,0,2025-01-01 02:46:00,2025-01-01 02:45:00,2025-01-01 02:47:00,False
2,2025-01-01 02:47:00,90.000000,36.153846,1,2025-01-01 02:48:00,2025-01-01 02:46:00,2025-01-01 02:48:00,False
3,2025-01-01 02:48:00,90.000000,39.230769,1,2025-01-01 02:49:00,2025-01-01 02:48:00,2025-01-01 02:49:00,False
4,2025-01-01 02:49:00,90.000000,42.307692,1,2025-01-01 02:50:00,2025-01-01 02:49:00,2025-01-01 02:50:00,False
5,2025-01-01 02:50:00,90.000000,45.384615,1,2025-01-01 02:51:00,2025-01-01 02:50:00,2025-01-01 02:51:00,False
...,...,...,...,...,...,...,...,...
21,2025-01-01 06:01:00,57.486133,48.946371,0,2025-01-01 06:02:00,2025-01-01 06:00:00,2025-01-01 06:03:00,False
198,2025-01-01 06:03:00,270.000000,33.076923,1,2025-01-01 06:04:00,2025-01-01 06:02:00,2025-01-01 06:05:00,False
22,2025-01-01 06:05:00,59.685858,35.627086,0,2025-01-01 06:06:00,2025-01-01 06:04:00,2025-01-01 06:07:00,False
202,2025-01-01 06:07:00,90.000000,36.153846,1,2025-01-01 06:08:00,2025-01-01 06:06:00,2025-01-01 06:09:00,False


In [14]:
master_sch_df[master_sch_df["exp_num"] == 1]

,stt_tstmp_us,pointing_az,pointing_el,exp_num,end_time,allowed_start_time,allowed_end_time,is_overlaped
2,2025-01-01 02:47:00,90.0,36.153846,1,2025-01-01 02:48:00,2025-01-01 02:46:00,2025-01-01 02:48:00,False
3,2025-01-01 02:48:00,90.0,39.230769,1,2025-01-01 02:49:00,2025-01-01 02:48:00,2025-01-01 02:49:00,False
4,2025-01-01 02:49:00,90.0,42.307692,1,2025-01-01 02:50:00,2025-01-01 02:49:00,2025-01-01 02:50:00,False
5,2025-01-01 02:50:00,90.0,45.384615,1,2025-01-01 02:51:00,2025-01-01 02:50:00,2025-01-01 02:51:00,False
6,2025-01-01 02:51:00,90.0,48.461538,1,2025-01-01 02:52:00,2025-01-01 02:51:00,2025-01-01 02:52:00,False
...,...,...,...,...,...,...,...,...
186,2025-01-01 05:51:00,270.0,70.000000,1,2025-01-01 05:52:00,2025-01-01 05:50:00,2025-01-01 05:53:00,False
190,2025-01-01 05:55:00,270.0,57.692308,1,2025-01-01 05:56:00,2025-01-01 05:54:00,2025-01-01 05:57:00,False
194,2025-01-01 05:59:00,270.0,45.384615,1,2025-01-01 06:00:00,2025-01-01 05:58:00,2025-01-01 06:01:00,False
198,2025-01-01 06:03:00,270.0,33.076923,1,2025-01-01 06:04:00,2025-01-01 06:02:00,2025-01-01 06:05:00,False


In [15]:
master_sch = Schedule.from_dataframe(master_sch_df, master_sch_meta)
master_sch

Schedule(meta={0: ExperimentDetail(id=0, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(60000000,'us')), 1: ExperimentDetail(id=1, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(60000000,'us'))}, stt_tstmp_us=array(['2025-01-01T02:45:00.000000', '2025-01-01T02:47:00.000000',
       '2025-01-01T02:48:00.000000', '2025-01-01T02:49:00.000000',
       '2025-01-01T02:50:00.000000', '2025-01-01T02:51:00.000000',
       '2025-01-01T02:52:00.000000', '2025-01-01T02:53:00.000000',
       '2025-01-01T02:54:00.000000', '2025-01-01T02:55:00.000000',
       '2025-01-01T02:56:00.000000', '2025-01-01T02:57:00.000000',
       '2025-01-01T02:58:00.000000', '2025-01-01T02:59:00.000000',
       '2025-01-01T03:00:00.000000', '2025-01-01T03:01:00.000000',
       '2025-01-01T03:02:00.0

In [16]:
# plots.schedule_plot(master_sch)

In [ ]:
# plot = plots.schedule_plot_bokeh(master_sch)

from sorts.plotting_deps import bp, bokeh_models

schedule = master_sch
start_time = None
end_time = None

df = schedule.as_dataframe()

start_time = start_time if start_time is not None else df[schedule.cn.start_time].min()
end_time = end_time if end_time is not None else df[schedule.cn.start_time].max()
df = df[(df[schedule.cn.start_time] >= start_time) & (df[schedule.cn.end_time] <= end_time)]

# bokeh requires str type for categorical axis
df[schedule.cn.exp_num] = df[schedule.cn.exp_num].astype(str)

bar = bp.figure(
    y_range=df[schedule.cn.exp_num].unique(),  # type: ignore
    x_axis_type="datetime",
    x_axis_location="above",
    width=800,
    height=300,
)
bar.add_tools(bokeh_models.HoverTool())

bar.hbar(
    y=df[schedule.cn.exp_num], left=df[schedule.cn.start_time], right=df[schedule.cn.end_time]  # type: ignore
)

minimap = bp.figure(
    title="Drag the middle and edges of the selection box to change the range above",
    height=130,
    width=800,
    # x_range=bar.x_range,
    x_axis_type="datetime",
    y_axis_type=None,
    tools="",
    toolbar_location=None,
)
minimap.x_range.range_padding = 0  # type: ignore
minimap.x_range.bounds = "auto"  # type: ignore

# NOTE: a dummy line is plotted; select tool doesn't work well without any data plotted
minimap.line(x=[df[schedule.cn.start_time].min(), df[schedule.cn.start_time].max()], y=[0, 0])
minimap_range_tool = bokeh_models.RangeTool(x_range=bar.x_range, start_gesture="pan")
minimap.add_tools(minimap_range_tool)

plot = bp.column(bar, minimap)

bp.show(plot)

In [28]:
from datetime import datetime
from sorts.plotting_deps import bp, bokeh_models
from sorts.schedule_v2 import Schedule

def schedule_plot_bokeh(
    schedule: Schedule, start_time: datetime | None = None, end_time: datetime | None = None
):
    """
    Note:
    Plotting the full schedule can be computationally demanding and lead to application crashes.
    Limiting the plot range by `start_time` and `end_time` param is recommended.

    A good starting point is a 1 hour time range.
    """

    df = schedule.as_dataframe()

    start_time = start_time if start_time is not None else df[schedule.cn.start_time].min()
    end_time = end_time if end_time is not None else df[schedule.cn.start_time].max()
    df = df[(df[schedule.cn.start_time] >= start_time) & (df[schedule.cn.end_time] <= end_time)]

    # bokeh requires str type for categorical axis
    df[schedule.cn.exp_num] = df[schedule.cn.exp_num].astype(str)

    bar = bp.figure(
        y_range=df[schedule.cn.exp_num].unique(),  # type: ignore
        x_axis_type="datetime",
        x_axis_location="above",
        width=800,
        height=300,
    )
    bar.add_tools(bokeh_models.HoverTool())

    bar.hbar(
        y=df[schedule.cn.exp_num], left=df[schedule.cn.start_time], right=df[schedule.cn.end_time]  # type: ignore
    )

    minimap = bp.figure(
        title="Drag the middle and edges of the selection box to change the range above",
        height=130,
        width=800,
        # x_range=bar.x_range,
        x_axis_type="datetime",
        y_axis_type=None,
        tools="",
        toolbar_location=None,
    )
    minimap.x_range.range_padding = 0  # type: ignore
    minimap.x_range.bounds = "auto"  # type: ignore

    # NOTE: a dummy line is plotted; select tool doesn't work well without any data plotted
    minimap.line(x=[df[schedule.cn.start_time].min(), df[schedule.cn.start_time].max()], y=[0, 0])
    minimap_range_tool = bokeh_models.RangeTool(x_range=bar.x_range, start_gesture="pan")
    minimap.add_tools(minimap_range_tool)

    plot = bp.column(bar, minimap)

    return plot

In [54]:
start_datetime_widget = widgets.DatetimePicker(
    value=df[schedule.cn.start_time].min().tz_localize("utc"),
    description='Start Time',
)
end_datetime_widget = widgets.DatetimePicker(
    value=df[schedule.cn.start_time].min().tz_localize("utc") + np.timedelta64(1, "h"),
    description='End Time',
)

date_range_widget = widgets.HBox([start_datetime_widget, end_datetime_widget])
date_range_widget

In [55]:
# TODO: leverage `notebook_handle`, e.g. `plot_nbh = bp.show(plot, notebook_handle=True)` ?
plot = schedule_plot_bokeh(master_sch, start_datetime_widget.value.replace(tzinfo=None), end_datetime_widget.value.replace(tzinfo=None))
bp.show(plot)